# Cytosolic Small-Molecule Buffering Analysis

This notebook investigates whether cytosolic small molecules (GSH, ATP, etc.) can account for
the **deficit** between total cellular metal measured by ICP-MS and the metal accounted for
by the proteome model in Figure 1.

## Rationale

The reviewer noted that the absence of *free* (hydrated) metal ions does not mean the absence
of an *exchangeable* labile pool: small molecules such as glutathione (GSH) and ATP buffer
transition metals at µM concentrations in the cytosol (Simm et al. 2021, JBIC).

The analysis proceeds in three steps (matching the todo list in the plan):

1. **Deficit calculation** – compute `deficit_uM = total_ICP − protein_sim`, converted to µM
   using simulation-derived cell volumes.
2. **Approach 1** – given the deficit and known small-molecule concentrations, estimate the
   minimum effective Kd a hypothetical single ligand would need to sequester the observed deficit.
3. **Approach 2** – use published stability constants to predict the labile pool and compare to
   the deficit.

In [1]:
import os
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.ticker as ticker

from wholecell.utils import units
from ecoli.library.schema import bulk_name_to_idx
os.chdir(os.path.expanduser('~/dev/vEcoli/notebooks/cofactors'))
os.makedirs("buffer_figures", exist_ok=True)

plt.rcParams.update({
    "font.size": 11,
    "axes.titlesize": 12,
    "axes.labelsize": 11,
    "figure.dpi": 120,
})

## 1. Load metals plot data

Columns:
- `Atoms/cell (experiment)` – ICP-MS total cellular metal  
- `Atoms/cell (simulation)` – model prediction (protein-bound metal only)  
- `Medium` – `"Rich"` or `"Minimal"`

In [2]:
df = pd.read_csv("data/metals_plot_data.csv")

# Strip stray quotes introduced by the quoting style in the CSV
df["Element"] = df["Element"].str.replace('"', "").str.strip()
df["Medium"]  = df["Medium"].str.replace('"', "").str.strip()

df

,Element,Atoms/cell (experiment),"Atoms/cell (experiment), stddev",Atoms/cell (simulation),"Atoms/cell (simulation), stddev",Medium
0,CU,87576.668290,29368.575624,15211.081627,1051.768662,Rich
1,MO,10797.032626,1000.000000,3444.940736,394.060051,Rich
2,NI,3240.269333,1000.000000,1284.841822,176.780770,Rich
3,ZN,399640.415582,88105.726872,357249.636639,22782.838522,Rich
4,CO,443.558908,100.000000,0.000000,0.000000,Rich
5,MN,46743.626407,12922.173275,69398.444095,3754.988890,Rich
6,FE,607143.184501,128046.989721,494684.179294,34074.181705,Rich
7,V,249.432594,1000.000000,0.000000,0.000000,Rich
8,CR,3414.351358,1000.000000,0.000000,0.000000,Rich
9,FE,179978.374255,24669.603524,267598.232091,16974.659764,Minimal


## 2. Compute the deficit

$$
\text{deficit}_{\text{atoms}} = N_{\text{ICP}} - N_{\text{sim}}
$$

In [3]:
df["deficit_atoms"] = df["Atoms/cell (experiment)"] - df["Atoms/cell (simulation)"]

# Propagate uncertainties in quadrature
df["deficit_atoms_err"] = np.sqrt(
    df["Atoms/cell (experiment), stddev"]**2 +
    df["Atoms/cell (simulation), stddev"]**2
)

# ---- From final_paper_validation.ipynb: manually copied over the
# average cell volumes per condition used for the conversion to µM ----
cell_volume_fL = {
    "Rich":    3.073264,
    "Minimal": 1.44983
} # units in fL

COUNTS_UNITS = units.mol
VOLUME_UNITS = units.L
CONC_UNITS = COUNTS_UNITS / VOLUME_UNITS

def atoms_to_uM(atoms_per_cell: float, volume_fL: float) -> float:
    """Convert atoms/cell to µM given cell volume in femtolitres."""
    n_avogadro = 6.02214076e23 / COUNTS_UNITS
    volume_L = VOLUME_UNITS * volume_fL * 1e-15
    molar    = atoms_per_cell / (n_avogadro * volume_L)

    return molar.asUnit(CONC_UNITS)  # M

In [4]:
# Convert to µM using per-condition cell volume
df["cell_volume_fL"] = df["Medium"].map(cell_volume_fL)
df["deficit_M"]     = df.apply(
    lambda r: atoms_to_uM(r["deficit_atoms"], r["cell_volume_fL"]), axis=1
)
df["deficit_M_err"] = df.apply(
    lambda r: atoms_to_uM(r["deficit_atoms_err"], r["cell_volume_fL"]), axis=1
)

# Also convert total and sim to µM for later use
df["total_M"] = df.apply(
    lambda r: atoms_to_uM(r["Atoms/cell (experiment)"], r["cell_volume_fL"]), axis=1
)
df["sim_M"] = df.apply(
    lambda r: atoms_to_uM(r["Atoms/cell (simulation)"], r["cell_volume_fL"]), axis=1
)

In [5]:
# Focus on the six biologically relevant transition metals
TARGET_METALS = ["FE", "ZN", "MN", "CU", "MO", "NI"]
df_metals = df[df["Element"].isin(TARGET_METALS)].copy()

display_cols = [
    "Element", "Medium",
    "total_M", "sim_M",
    "deficit_atoms", "deficit_M", "deficit_M_err",
]
df_metals[display_cols].sort_values(["Element", "Medium"]).reset_index(drop=True)

,Element,Medium,total_M,sim_M,deficit_atoms,deficit_M,deficit_M_err
0,CU,Minimal,8.991643832170565e-06 [mol/L],9.728728786244183e-06 [mol/L],-643.554795,-7.370849540736187e-07 [mol/L],1.5124513402886703e-06 [mol/L]
1,CU,Rich,4.7319227722860776e-05 [mol/L],8.218817288590833e-06 [mol/L],72365.586663,3.910041043426994e-05 [mol/L],1.5878535219571316e-05 [mol/L]
2,FE,Minimal,0.00020613528599688457 [mol/L],0.0003064892564602468 [mol/L],-87619.857836,-0.00010035397046336227 [mol/L],3.4297501263943724e-05 [mol/L]
3,FE,Rich,0.00032805023493987146 [mol/L],0.0002672866391011376 [mol/L],112459.005207,6.0763595838733856e-05 [mol/L],7.15937961292907e-05 [mol/L]
4,MN,Rich,2.525641070542673e-05 [mol/L],3.749721065968548e-05 [mol/L],-22654.817688,-1.2240799954258754e-05 [mol/L],7.2708883238793416e-06 [mol/L]
5,MO,Minimal,4.56223543910277e-06 [mol/L],2.50833222258943e-06 [mol/L],1793.279399,2.0539032165133402e-06 [mol/L],1.1876101771072852e-06 [mol/L]
6,MO,Rich,5.833828296242571e-06 [mol/L],1.8613626021801658e-06 [mol/L],7352.091890,3.972465694062405e-06 [mol/L],5.807557002002923e-07 [mol/L]
7,NI,Minimal,4.329449437680053e-06 [mol/L],7.835023416485401e-07 [mol/L],3095.994897,3.545947096031513e-06 [mol/L],1.1543519430757217e-06 [mol/L]
8,NI,Rich,1.7507750117177837e-06 [mol/L],6.94222832978171e-07 [mol/L],1955.427511,1.0565521787396126e-06 [mol/L],5.486956448007572e-07 [mol/L]
9,ZN,Minimal,0.0002288470937471057 [mol/L],0.0002462679884360221 [mol/L],-15210.323109,-1.7420894688916387e-05 [mol/L],6.532513329550545e-05 [mol/L]


## 3. Approach One: Prediction of Labile Pool by using exisitng small molecule concentration and $K_D$ reported in literature

Given published stability constants (log K₁) and ligand concentrations, predict what
fraction of the **deficit** is sequestered by small molecules.

**Competition model** (1:1 binding, ligands in large excess over metal):

$$
[\text{M}_\text{free}] = \frac{[\text{M}_\text{deficit}]}{1 + \sum_i K_{A,i} [L_{free,i}]}
\qquad K_{A,i} = 10^{\log \beta}
$$

$$
[M_{\text{deficit}}]_{estimated} = [\text{M}_\text{free}] [1 + \sum_i K_{A,i} [L_{free,i}]]
$$

$$
[M_{\text{deficit}}]_{measured-ish} = [\text{M}_\text{exp}] - [\text{M}_\text{cofacter-bound}]
$$

Ligand concentrations: simulation-derived small molecules concentrations

Consider competition amongst the small molecule ligands as the following:

$$
[\text{L}_\text{free}] =\frac{[L_{\text{listener\_bulk}}]}{1 + \sum_i K_{A,i} [M_{free,i}]}
$$

![image.png](attachment:a274feba-e73f-4eed-828a-28fb08c01fd6.png)

### load sim

In [6]:
import dill
def load_sim(
        folder_path:str,
):
    """ This function is meant to load an output of a simulation in timeseries form.
        Note: This is not designed for parquet output format.
    """
    # --- Load Sim ---
    output = np.load(folder_path + '0_output.npy',allow_pickle='TRUE').item()
    output = output['agents']['0']
    fba = output['listeners']['fba_results']
    bulk = pd.DataFrame(output['bulk'])
    f = open(folder_path + 'agent_steps.pkl', 'rb')
    agent = dill.load(f)
    f.close()

    metabolism = agent['ecoli-metabolism-redux-classic']

    return fba, bulk, metabolism, output

In [7]:
os.chdir(os.path.expanduser('~/dev/vEcoli'))
[fba_basal, bulk_basal, metabolism, output_basal] = load_sim(
    folder_path="out_cyrus/basal/metabolism_classic_1500_2026-03-24/"
)
[fba_rich, bulk_rich, metabolism, output_rich] = load_sim(
    folder_path="out_cyrus/with_aa/metabolism_classic_1500_2026-03-24/"
)

### Consider Metal Ion Competition for Small Molecules First

In [1]:
import pandas as pd
import numpy as np

df = pd.read_csv("notebooks/cofactors/data/buffer_constants.csv", index_col=0)

# --- Free metal concentrations ---
free_metal_measured_minimal = df.iloc[:, 0].to_dict()  # M9 cells column M
free_metal_measured_rich    = df.iloc[:, 1].to_dict()  # LB cells column M

# --- Experimental buffered metal concentrations ---
buffered_metal_measured_minimal = df.iloc[:, 2].to_dict()  # M9 cells column M
buffered_metal_measured_minimal_std    = df.iloc[:, 3].to_dict()  # std M


# --- Buffer to metal dictionary (skip first two columns) ---
buffer_df = df.iloc[:, 4:]  # all ligand columns

buffer_to_metal = {}
for buffer in buffer_df.columns:
    entries = {}
    for metal in buffer_df.index:
        val = buffer_df.loc[metal, buffer]
        if pd.notna(val) and val != "":
            entries[metal] = float(val)
    if entries:  # only include buffer if it has at least one entry
        buffer_to_metal[buffer] = entries

metal_to_buffer = {}
for buffer, metals in buffer_to_metal.items():
    for metal, logbeta in metals.items():
        if metal not in metal_to_buffer:
            metal_to_buffer[metal] = {}
        metal_to_buffer[metal][buffer] = logbeta

# Sanity check
print("Metals in minimal media:", free_metal_measured_minimal)
print("Metals in rich media:   ", free_metal_measured_rich)
print("\nBuffers found:", list(buffer_to_metal.keys()))

FileNotFoundError: [Errno 2] No such file or directory: 'notebooks/cofactors/data/buffer_constants.csv'

In [9]:
metals = list(free_metal_measured_minimal.keys())
conditions = ["Minimal", "Rich"]
metal_to_free_buffer= dict({
    metal: dict() for metal in metals
})
metal_to_buffered_conc = dict({
    cond: dict() for cond in conditions
})

for cond in conditions:
    if cond == "Minimal":
        bulk = bulk_basal
    else:
        bulk = bulk_rich
    for metal_of_interest in metals:
        print(f'Analyzing {metal_of_interest}...')
        buffer_mol = list(metal_to_buffer[metal_of_interest].keys()) # ['GLUTATHIONE[c]', 'CYS[c]']
        buffer_idx = bulk_name_to_idx(buffer_mol, metabolism.bulk_ids)
        buffer_df =  pd.DataFrame({"WC Mean Count":
                                   pd.DataFrame(bulk[buffer_idx]).mean(axis=0).values}, index=buffer_mol)
        buffer_df['WC Conc (M)'] = buffer_df['WC Mean Count'].apply(
        lambda x: atoms_to_uM(x, cell_volume_fL[cond]).asNumber()
        )
        print(buffer_df)

        # calculate free buffer concentration considering competition from other metals
        for buffer in buffer_mol:
            total_buffer = buffer_df.loc[buffer, 'WC Conc (M)']
            competing_metals = buffer_to_metal.get(buffer, {})
            competition_factor = sum(
                10 ** logbeta * free_metal_measured_minimal.get(metal, 0)
                for metal, logbeta in competing_metals.items()
            )
            free_buffer = total_buffer / (1 + competition_factor)
            metal_to_free_buffer[metal_of_interest][buffer] = free_buffer
            print(f"{metal_of_interest} -- {buffer}: total_buffer={total_buffer:.2e} M, free_buffer={free_buffer:.2e} M")

        # calculate buffered metal concentration
        competing_buffer = metal_to_buffer.get(metal_of_interest, {})
        competition_factor = sum(10**logbeta * metal_to_free_buffer[metal_of_interest][buffer]
            for buffer, logbeta in competing_buffer.items()
        )
        buffered_metal = free_metal_measured_minimal[metal_of_interest] * (1 + competition_factor)
        print(f"Buffered {metal_of_interest} in {cond} condition: {buffered_metal:.2e} M")
        metal_to_buffered_conc[cond][metal_of_interest] = buffered_metal

Analyzing MN...
                    WC Mean Count  WC Conc (M)
GLUTATHIONE[c]       8.548057e+06     0.009790
HIS[c]               5.325575e+04     0.000061
CIT[c]               1.333817e+06     0.001528
ATP[c]               6.999812e+06     0.008017
TRP[c]               1.577320e+04     0.000018
MET[c]               5.899806e+04     0.000068
L-ALPHA-ALANINE[c]   8.109509e+05     0.000929
LEU[c]               1.104851e+05     0.000127
VAL[c]               2.216971e+05     0.000254
GLY[c]               4.532069e+05     0.000519
MN -- GLUTATHIONE[c]: total_buffer=9.79e-03 M, free_buffer=1.21e-06 M
MN -- HIS[c]: total_buffer=6.10e-05 M, free_buffer=5.67e-05 M
MN -- CIT[c]: total_buffer=1.53e-03 M, free_buffer=1.50e-03 M
MN -- ATP[c]: total_buffer=8.02e-03 M, free_buffer=6.03e-03 M
MN -- TRP[c]: total_buffer=1.81e-05 M, free_buffer=1.80e-05 M
MN -- MET[c]: total_buffer=6.76e-05 M, free_buffer=6.70e-05 M
MN -- L-ALPHA-ALANINE[c]: total_buffer=9.29e-04 M, free_buffer=9.25e-04 M
MN -- LEU[c]:

In [10]:
import plotly.graph_objects as go

# --- Filter to positive deficits only ---
# deficit_M values are Unum; .str.replace(...).astype(float) yields NaN — use .asNumber().
def _deficit_m_to_float(ser):
    return ser.apply(lambda x: x.asNumber())


deficit_m_float = _deficit_m_to_float(df_metals["deficit_M"])
deficit_pos = df_metals[deficit_m_float > 0].copy()
deficit_pos["deficit_val"] = _deficit_m_to_float(deficit_pos["deficit_M"])

# --- Build a long-form dataframe from metal_to_buffered_conc ---
rows = []
for medium, metals in metal_to_buffered_conc.items():
    for element, buffered_conc in metals.items():
        rows.append(
            {"Medium": medium, "Element": element, "buffered_conc": float(buffered_conc)}
        )
buffered_df = pd.DataFrame(rows)

# --- Merge on Medium and Element ---
plot_df = deficit_pos.merge(buffered_df, on=["Medium", "Element"], how="inner")

# --- Plot ---
fig = go.Figure()

colors = {"Minimal": "steelblue", "Rich": "tomato"}
symbols = {"Minimal": "circle", "Rich": "diamond"}

for medium, group in plot_df.groupby("Medium"):
    fig.add_trace(go.Scatter(
        x=group["buffered_conc"] * 1E6,
        y=group["deficit_val"] * 1E6,
        mode="markers+text",
        name=medium,
        text=group["Element"],
        textposition="top center",
        marker=dict(
            size=12,
            color=colors[medium],
            symbol=symbols[medium],
            line=dict(width=1, color="black")
        )
    ))

fig.update_layout(
    title="Buffered Metal Concentration vs. Measured Deficit - With Competition",
    xaxis=dict(title="Buffered Metal Concentration [umol/L]", type="log", range=[-10, 10]),
    yaxis=dict(title="Deficit [umol/L]", type="log", range=[-10, 10]),
    legend_title="Medium",
    template="plotly_white",
    width=800,
    height=550
)

fig.show()

### Simplify the model to ignore small molecule competition

In [11]:
metals = list(free_metal_measured_minimal.keys())
small_molecules = list(buffer_to_metal.keys())

conditions = ["Minimal", "Rich"]
metal_to_free_buffer= dict({
    metal: dict() for metal in metals
})
metal_to_buffered_conc_simplified = dict({
    cond: dict() for cond in conditions
})

for cond in conditions:
    if cond == "Minimal":
        bulk = bulk_basal
    else:
        bulk = bulk_rich
    for metal_of_interest in metals:
        print(f'Analyzing {metal_of_interest}...')
        buffer_mol = list(metal_to_buffer[metal_of_interest].keys()) # ['GLUTATHIONE[c]', 'CYS[c]']
        buffer_idx = bulk_name_to_idx(buffer_mol, metabolism.bulk_ids)
        buffer_df =  pd.DataFrame({"WC Mean Count":
                                   pd.DataFrame(bulk[buffer_idx]).mean(axis=0).values}, index=buffer_mol)
        buffer_df['WC Conc (M)'] = buffer_df['WC Mean Count'].apply(
        lambda x: atoms_to_uM(x, cell_volume_fL[cond]).asNumber()
        )
        print(buffer_df)

        # calculate buffered metal concentration
        competing_buffer = metal_to_buffer.get(metal_of_interest, {})
        competition_factor = sum(10**logbeta * buffer_df.loc[buffer, 'WC Conc (M)']
            for buffer, logbeta in competing_buffer.items()
        )
        buffered_metal = free_metal_measured_minimal[metal_of_interest] * (1 + competition_factor)
        print(f"Buffered {metal_of_interest} in {cond} condition: {buffered_metal:.2e} M")
        metal_to_buffered_conc_simplified[cond][metal_of_interest] = buffered_metal

Analyzing MN...
                    WC Mean Count  WC Conc (M)
GLUTATHIONE[c]       8.548057e+06     0.009790
HIS[c]               5.325575e+04     0.000061
CIT[c]               1.333817e+06     0.001528
ATP[c]               6.999812e+06     0.008017
TRP[c]               1.577320e+04     0.000018
MET[c]               5.899806e+04     0.000068
L-ALPHA-ALANINE[c]   8.109509e+05     0.000929
LEU[c]               1.104851e+05     0.000127
VAL[c]               2.216971e+05     0.000254
GLY[c]               4.532069e+05     0.000519
Buffered MN in Minimal condition: 2.62e-03 M
Analyzing FE...
                WC Mean Count  WC Conc (M)
GLUTATHIONE[c]   8.548057e+06     0.009790
CYS[c]           1.577320e+04     0.000018
HIS[c]           5.325575e+04     0.000061
CIT[c]           1.333817e+06     0.001528
ATP[c]           6.999812e+06     0.008017
TRP[c]           1.577320e+04     0.000018
Buffered FE in Minimal condition: 1.64e-04 M
Analyzing CO...
                    WC Mean Count  WC Conc (

In [12]:
import plotly.graph_objects as go

# --- Filter to positive deficits only ---
# deficit_M values are Unum; .str.replace(...).astype(float) yields NaN — use .asNumber().
def _deficit_m_to_float(ser):
    return ser.apply(lambda x: x.asNumber())


deficit_m_float = _deficit_m_to_float(df_metals["deficit_M"])
deficit_pos = df_metals[deficit_m_float > 0].copy()
deficit_pos["deficit_val"] = _deficit_m_to_float(deficit_pos["deficit_M"])

# --- Build a long-form dataframe from metal_to_buffered_conc ---
rows = []
for medium, metals in metal_to_buffered_conc_simplified.items():
    for element, buffered_conc in metals.items():
        rows.append(
            {"Medium": medium, "Element": element, "buffered_conc": float(buffered_conc)}
        )
buffered_df = pd.DataFrame(rows)

# --- Merge on Medium and Element ---
plot_df = deficit_pos.merge(buffered_df, on=["Medium", "Element"], how="inner")

# --- Plot ---
fig2 = go.Figure()

colors = {"Minimal": "steelblue", "Rich": "tomato"}
symbols = {"Minimal": "circle", "Rich": "diamond"}

for medium, group in plot_df.groupby("Medium"):
    fig2.add_trace(go.Scatter(
        x=group["buffered_conc"] * 1E6,
        y=group["deficit_val"] * 1E6,
        mode="markers+text",
        name=medium,
        text=group["Element"],
        textposition="top center",
        marker=dict(
            size=12,
            color=colors[medium],
            symbol=symbols[medium],
            line=dict(width=1, color="black")
        )
    ))

fig2.update_layout(
    title="Buffered Metal Concentration vs. Measured Deficit - Simplified",
    xaxis=dict(title="Buffered Metal Concentration [umol/L]", type="log", range=[-10, 10]),
    yaxis=dict(title="Deficit [umol/L]", type="log", range=[-10, 10]),
    legend_title="Medium",
    template="plotly_white",
    width=800,
    height=550
)

fig2.show()

In [13]:
fig.show()

In [36]:
def build_plot_df(buffered_conc_dict):
    rows = []
    for medium, metals in buffered_conc_dict.items():
        for element, buffered_conc in metals.items():
            rows.append({"Medium": medium, "Element": element, "buffered_conc": float(buffered_conc)})
    return deficit_pos.merge(pd.DataFrame(rows), on=["Medium", "Element"], how="inner")

plot_df_simplified  = build_plot_df(metal_to_buffered_conc_simplified)
plot_df_competition = build_plot_df(metal_to_buffered_conc)

colors  = {"Minimal": "steelblue", "Rich": "tomato"}
symbols = {"Minimal": "circle",    "Rich": "diamond"}

fig = go.Figure()

# --- Add y=x reference line ---
x_ref = np.logspace(-20, 10, num=100)
fig.add_trace(go.Scatter(
    x=x_ref * 1e6,
    y=x_ref * 1e6,
    mode="lines",
    name="y = x",
    line=dict(color="gray", dash="dash"),
    showlegend=False,
))

# --- Simplified points (alpha = 0.7, hollow-ish) ---
for medium, group in plot_df_simplified.groupby("Medium"):
    rgba = "rgba(70,130,180,0.7)"  if medium == "Minimal" else "rgba(255,99,71,0.7)"
    fig.add_trace(go.Scatter(
        x=group["buffered_conc"] * 1e6,
        y=group["deficit_val"]   * 1e6,
        mode="markers",
        name=f"{medium} (simplified)",
        marker=dict(size=12, color=rgba, symbol=symbols[medium],
                    line=dict(width=1, color="black")),
        legendgroup=medium,
    ))

# --- Competition points (alpha = 1.0, labelled) ---
for medium, group in plot_df_competition.groupby("Medium"):
    fig.add_trace(go.Scatter(
        x=group["buffered_conc"] * 1e6,
        y=group["deficit_val"]   * 1e6,
        mode="markers+text",
        name=f"{medium} (competition)",
        text=group["Element"],
        textposition="top center",
        marker=dict(size=12, color=colors[medium], symbol=symbols[medium],
                    line=dict(width=1, color="black"), opacity=1.0),
        legendgroup=medium,
    ))

# --- Arrows from simplified → competition for each (Medium, Element) pair ---
merged = plot_df_simplified.merge(
    plot_df_competition,
    on=["Medium", "Element"],
    suffixes=("_simp", "_comp")
)

for _, row in merged.iterrows():
    # Use raw data coordinates; Plotly handles log transform from axis type="log".
    x0 = row["buffered_conc_simp"] * 1e6
    y0 = row["deficit_val_simp"]   * 1e6
    x1 = row["buffered_conc_comp"] * 1e6
    y1 = row["deficit_val_comp"]   * 1e6

    vals = np.array([x0, y0, x1, y1], dtype=float)
    if (not np.isfinite(vals).all()) or (vals <= 0).any():
        continue

    # Skip if essentially no shift
    if abs(x1 - x0) < 1e-30 and abs(y1 - y0) < 1e-30:
        continue

    fig.add_annotation(
        x=np.log10(x1), y=np.log10(y1),          # arrowhead at competition point
        ax=np.log10(x0), ay=np.log10(y0),        # arrow tail at simplified point
        xref="x", yref="y",
        axref="x", ayref="y",
        showarrow=True,
        arrowhead=3,
        arrowsize=1.2,
        arrowwidth=1.5,
        arrowcolor=colors[row["Medium"]],
        opacity=0.7,
        text="",
    )

fig.update_layout(
    title="Buffered Metal Concentration vs. Deficit<br><sup>Arrows: Simplified → With Competition</sup>",
    xaxis=dict(title="Buffered Metal Concentration [µmol/L]", type="log"),
    yaxis=dict(title="Deficit [µmol/L]",                      type="log"),
    legend_title="Medium / Model",
    template="plotly_white",
    width=900,
    height=600,
)

fig.update_xaxes(range=[-10, 10])
fig.update_yaxes(range=[-10, 10])

# fig.show(renderer="browser")
fig.write_image('notebooks/cofactors/buffer_figures/buffered_conc_vs_deficit_with_arrows.svg', scale=3)

## Plot deficit versus experimentally measured deficit (buffered concentration)

In [84]:
df_metals_adj

,index,Element,Atoms/cell (experiment),"Atoms/cell (experiment), stddev",Atoms/cell (simulation),"Atoms/cell (simulation), stddev",Medium,deficit_atoms,deficit_atoms_err,cell_volume_fL,deficit_M,deficit_M_err,total_M,sim_M
0,0,CU,87576.668290,29368.575624,15211.081627,1051.768662,Rich,72365.586663,29387.402939,3.073264,3.910041e-05,1.587854e-05,4.7319227722860776e-05 [mol/L],8.218817288590833e-06 [mol/L]
1,1,MO,10797.032626,1000.000000,3444.940736,394.060051,Rich,7352.091890,1074.841069,3.073264,3.972466e-06,5.807557e-07,5.833828296242571e-06 [mol/L],1.8613626021801658e-06 [mol/L]
2,2,NI,3240.269333,1000.000000,1284.841822,176.780770,Rich,1955.427511,1015.505510,3.073264,1.056552e-06,5.486956e-07,1.7507750117177837e-06 [mol/L],6.94222832978171e-07 [mol/L]
3,3,ZN,399640.415582,88105.726872,357249.636639,22782.838522,Rich,42390.778943,91003.718818,3.073264,2.290449e-05,4.917092e-05,0.00021593280723540271 [mol/L],0.00019302831724600316 [mol/L]
4,5,MN,46743.626407,12922.173275,69398.444095,3754.988890,Rich,-22654.817688,13456.689924,3.073264,-1.224080e-05,7.270888e-06,2.525641070542673e-05 [mol/L],3.749721065968548e-05 [mol/L]
5,6,FE,607143.184501,128046.989721,494684.179294,34074.181705,Rich,112459.005207,132503.137455,3.073264,6.076360e-05,7.159380e-05,0.00032805023493987146 [mol/L],0.0002672866391011376 [mol/L]
6,9,FE,179978.374255,24669.603524,267598.232091,16974.659764,Minimal,-87619.857836,29945.423893,1.449830,-1.003540e-04,3.429750e-05,0.00020613528599688457 [mol/L],0.0003064892564602468 [mol/L]
7,10,CU,7850.676467,1000.000000,8494.231262,862.443333,Minimal,-643.554795,1320.533416,1.449830,1.000000e-10,1.512451e-06,8.991643832170565e-06 [mol/L],9.728728786244183e-06 [mol/L]
8,12,NI,3780.077086,1000.000000,684.082189,125.737464,Minimal,3095.994897,1007.873955,1.449830,3.545947e-06,1.154352e-06,4.329449437680053e-06 [mol/L],7.835023416485401e-07 [mol/L]
9,13,ZN,199808.236064,55212.922173,215018.559173,14304.788166,Minimal,-15210.323109,57035.898690,1.449830,1.000000e-10,6.532513e-05,0.0002288470937471057 [mol/L],0.0002462679884360221 [mol/L]


In [91]:
buffered_metal_measured_minimal

{'MN': 1.4e-06,
 'FE': 8e-05,
 'CO': nan,
 'ZN': 1.3e-05,
 'NI': 1.5e-05,
 'CU': 1e-05}

In [93]:
df_metals_adj= df_metals.copy().reset_index()

# process deficit data so that we can plot more data on the figure
for col in ["deficit_M", "deficit_M_err"]:
    df_metals_adj[col] = df_metals_adj[col].apply(lambda x: x.asNumber())

for i in range(len(df_metals_adj)):
    deficit_M = df_metals_adj.loc[i, "deficit_M"]
    deficit_M_err = df_metals_adj.loc[i, "deficit_M_err"]
    if (deficit_M + deficit_M_err > 0) and (deficit_M < 0):
        df_metals_adj.loc[i, "deficit_M"] = 1E-10

df_metals_minimal = df_metals_adj[df_metals_adj["Medium"] == "Minimal"].copy()

df_plot = pd.DataFrame({
    "Element":              df_metals_minimal["Element"],
    "measured buffer (M)":  df_metals_minimal["Element"].map(buffered_metal_measured_minimal),
    "std buffer":           df_metals_minimal["Element"].map(buffered_metal_measured_minimal_std),
    "estimated buffer (M)": df_metals_minimal["deficit_M"],
    "std estimated buffer": df_metals_minimal["deficit_M_err"],
})

In [94]:
df_plot

,Element,measured buffer (M),std buffer,estimated buffer (M),std estimated buffer
6,FE,0.000080,0.000020,-1.003540e-04,0.000034
7,CU,0.000010,0.000002,1.000000e-10,0.000002
8,NI,0.000015,0.000002,3.545947e-06,0.000001
9,ZN,0.000013,0.000003,1.000000e-10,0.000065
10,MO,NaN,NaN,2.053903e-06,0.000001


In [95]:
df_metals

,Element,Atoms/cell (experiment),"Atoms/cell (experiment), stddev",Atoms/cell (simulation),"Atoms/cell (simulation), stddev",Medium,deficit_atoms,deficit_atoms_err,cell_volume_fL,deficit_M,deficit_M_err,total_M,sim_M
0,CU,87576.668290,29368.575624,15211.081627,1051.768662,Rich,72365.586663,29387.402939,3.073264,3.910041043426994e-05 [mol/L],1.5878535219571316e-05 [mol/L],4.7319227722860776e-05 [mol/L],8.218817288590833e-06 [mol/L]
1,MO,10797.032626,1000.000000,3444.940736,394.060051,Rich,7352.091890,1074.841069,3.073264,3.972465694062405e-06 [mol/L],5.807557002002923e-07 [mol/L],5.833828296242571e-06 [mol/L],1.8613626021801658e-06 [mol/L]
2,NI,3240.269333,1000.000000,1284.841822,176.780770,Rich,1955.427511,1015.505510,3.073264,1.0565521787396126e-06 [mol/L],5.486956448007572e-07 [mol/L],1.7507750117177837e-06 [mol/L],6.94222832978171e-07 [mol/L]
3,ZN,399640.415582,88105.726872,357249.636639,22782.838522,Rich,42390.778943,91003.718818,3.073264,2.290448998939954e-05 [mol/L],4.9170923928408456e-05 [mol/L],0.00021593280723540271 [mol/L],0.00019302831724600316 [mol/L]
5,MN,46743.626407,12922.173275,69398.444095,3754.988890,Rich,-22654.817688,13456.689924,3.073264,-1.2240799954258754e-05 [mol/L],7.2708883238793416e-06 [mol/L],2.525641070542673e-05 [mol/L],3.749721065968548e-05 [mol/L]
6,FE,607143.184501,128046.989721,494684.179294,34074.181705,Rich,112459.005207,132503.137455,3.073264,6.0763595838733856e-05 [mol/L],7.15937961292907e-05 [mol/L],0.00032805023493987146 [mol/L],0.0002672866391011376 [mol/L]
9,FE,179978.374255,24669.603524,267598.232091,16974.659764,Minimal,-87619.857836,29945.423893,1.449830,-0.00010035397046336227 [mol/L],3.4297501263943724e-05 [mol/L],0.00020613528599688457 [mol/L],0.0003064892564602468 [mol/L]
10,CU,7850.676467,1000.000000,8494.231262,862.443333,Minimal,-643.554795,1320.533416,1.449830,-7.370849540736187e-07 [mol/L],1.5124513402886703e-06 [mol/L],8.991643832170565e-06 [mol/L],9.728728786244183e-06 [mol/L]
12,NI,3780.077086,1000.000000,684.082189,125.737464,Minimal,3095.994897,1007.873955,1.449830,3.545947096031513e-06 [mol/L],1.1543519430757217e-06 [mol/L],4.329449437680053e-06 [mol/L],7.835023416485401e-07 [mol/L]
13,ZN,199808.236064,55212.922173,215018.559173,14304.788166,Minimal,-15210.323109,57035.898690,1.449830,-1.7420894688916387e-05 [mol/L],6.532513329550545e-05 [mol/L],0.0002288470937471057 [mol/L],0.0002462679884360221 [mol/L]


In [102]:
from sklearn.metrics import r2_score

# --- Drop NaN rows and reset index ---
df_plot_clean = df_plot.dropna(subset=["measured buffer (M)", "estimated buffer (M)"]).reset_index(drop=True)

x     = df_plot_clean["measured buffer (M)"].astype(float)
y     = df_plot_clean["estimated buffer (M)"].astype(float)
x_err = df_plot_clean["std buffer"].astype(float)
y_err = df_plot_clean["std estimated buffer"].astype(float)

# --- R² (only on matched finite pairs) ---
mask = (x>0) & (y>0)
r2 = r2_score(np.log10(y[mask]), np.log10(x[mask]))

# --- y = x reference line ---
xy_min = min(x.min(), y.min()) * 0.1
xy_max = max(x.max(), y.max()) * 10
ref_range = np.geomspace(xy_min, xy_max, 100)

fig = go.Figure()

# y = x line
fig.add_trace(go.Scatter(
    x=np.log10(ref_range),
    y=np.log10(ref_range),
    mode="lines",
    name="y = x",
    line=dict(color="gray", dash="dash", width=1.5),
))

# Scatter points with error bars
fig.add_trace(go.Scatter(
    x=x,
    y=y,
    mode="markers+text",
    name="Minimal",
    text=df_plot_clean["Element"],
    textposition="top center",
    marker=dict(size=10, color="steelblue", line=dict(width=1, color="black")),
    error_x=dict(type="data", array=x_err, visible=True, color="steelblue", thickness=1.5),
    error_y=dict(type="data", array=y_err, visible=True, color="steelblue", thickness=1.5),
))

# R² annotation
fig.add_annotation(
    x=0.05, y=0.95,
    xref="paper", yref="paper",
    text=f"R² = {r2:.3f}",
    showarrow=False,
    font=dict(size=14),
    bgcolor="white",
    bordercolor="gray",
    borderwidth=1,
)

fig.update_layout(
    title="Measured vs. Estimated Buffered Metal Concentration (Minimal Media)",
    xaxis=dict(title="Measured Buffered Concentration [M]", type="log", range=[-11, 0]),
    yaxis=dict(title="Estimated Buffered Concentration [M]", type="log", range=[-11, 0]),
    template="plotly_white",
    width=650,
    height=600,
    showlegend=True,
)

fig.show()

/Users/heenasaqib/dev/vEcoli/.venv/lib/python3.12/site-packages/numpy/_core/function_base.py:451: RuntimeWarning: invalid value encountered in log10
  log_stop = _nx.log10(stop)
/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_68857/2607754311.py:24: RuntimeWarning: invalid value encountered in log10
  x=np.log10(ref_range),
/var/folders/s9/gn2yly0s7rzgcc2xyvt7nsxm0000gr/T/ipykernel_68857/2607754311.py:25: RuntimeWarning: invalid value encountered in log10
  y=np.log10(ref_range),


In [47]:
buffer_mol = list(buffer_to_metal.keys()) # ['GLUTATHIONE[c]', 'CYS[c]']
buffer_idx = bulk_name_to_idx(buffer_mol, metabolism.bulk_ids)
buffer_df =  pd.DataFrame({"WC Mean Count":
                               pd.DataFrame(bulk_basal[buffer_idx]).mean(axis=0).values}, index=buffer_mol)
buffer_df['WC Conc (M)'] = buffer_df['WC Mean Count'].apply(
    lambda x: atoms_to_uM(x, cell_volume_fL[cond]).asNumber()
)

In [48]:
buffer_df

,WC Mean Count,WC Conc (M)
GLUTATHIONE[c],8.548057e+06,0.004619
CYS[c],1.577320e+04,0.000009
HIS[c],5.325575e+04,0.000029
OXIDIZED-GLUTATHIONE[c],9.794649e+05,0.000529
L-ASPARTATE[c],1.841176e+06,0.000995
CIT[c],1.333817e+06,0.000721
ATP[c],6.999812e+06,0.003782
TRP[c],1.577320e+04,0.000009
MET[c],5.899806e+04,0.000032
L-ALPHA-ALANINE[c],8.109509e+05,0.000438


In [53]:
800*1E-6

0.0007999999999999999

In [74]:
buffer_mol = ['TRP[P]', 'TRP[e]','CYS[P]', 'GLUTATHIONE[P]', 'GLUTATHIONE[c]','ATP[P]']
buffer_idx = bulk_name_to_idx(buffer_mol, metabolism.bulk_ids)
buffer_df =  pd.DataFrame({"WC Mean Count":
                               pd.DataFrame(bulk_rich[buffer_idx]).mean(axis=0).values}, index=buffer_mol)
buffer_df['WC Conc (M)'] = buffer_df['WC Mean Count'].apply(
    lambda x: atoms_to_uM(x, cell_volume_fL[cond]).asNumber()
)
buffer_df

,WC Mean Count,WC Conc (M)
TRP[P],6.947494e+05,0.000375
TRP[e],0.000000e+00,0.000000
CYS[P],8.190865e+04,0.000044
GLUTATHIONE[P],2.219460e+07,0.011992
GLUTATHIONE[c],2.219460e+07,0.011992
ATP[P],1.817466e+07,0.009820


In [75]:
df_metals

,Element,Atoms/cell (experiment),"Atoms/cell (experiment), stddev",Atoms/cell (simulation),"Atoms/cell (simulation), stddev",Medium,deficit_atoms,deficit_atoms_err,cell_volume_fL,deficit_M,deficit_M_err,total_M,sim_M
0,CU,87576.668290,29368.575624,15211.081627,1051.768662,Rich,72365.586663,29387.402939,3.073264,3.910041043426994e-05 [mol/L],1.5878535219571316e-05 [mol/L],4.7319227722860776e-05 [mol/L],8.218817288590833e-06 [mol/L]
1,MO,10797.032626,1000.000000,3444.940736,394.060051,Rich,7352.091890,1074.841069,3.073264,3.972465694062405e-06 [mol/L],5.807557002002923e-07 [mol/L],5.833828296242571e-06 [mol/L],1.8613626021801658e-06 [mol/L]
2,NI,3240.269333,1000.000000,1284.841822,176.780770,Rich,1955.427511,1015.505510,3.073264,1.0565521787396126e-06 [mol/L],5.486956448007572e-07 [mol/L],1.7507750117177837e-06 [mol/L],6.94222832978171e-07 [mol/L]
3,ZN,399640.415582,88105.726872,357249.636639,22782.838522,Rich,42390.778943,91003.718818,3.073264,2.290448998939954e-05 [mol/L],4.9170923928408456e-05 [mol/L],0.00021593280723540271 [mol/L],0.00019302831724600316 [mol/L]
5,MN,46743.626407,12922.173275,69398.444095,3754.988890,Rich,-22654.817688,13456.689924,3.073264,-1.2240799954258754e-05 [mol/L],7.2708883238793416e-06 [mol/L],2.525641070542673e-05 [mol/L],3.749721065968548e-05 [mol/L]
6,FE,607143.184501,128046.989721,494684.179294,34074.181705,Rich,112459.005207,132503.137455,3.073264,6.0763595838733856e-05 [mol/L],7.15937961292907e-05 [mol/L],0.00032805023493987146 [mol/L],0.0002672866391011376 [mol/L]
9,FE,179978.374255,24669.603524,267598.232091,16974.659764,Minimal,-87619.857836,29945.423893,1.449830,-0.00010035397046336227 [mol/L],3.4297501263943724e-05 [mol/L],0.00020613528599688457 [mol/L],0.0003064892564602468 [mol/L]
10,CU,7850.676467,1000.000000,8494.231262,862.443333,Minimal,-643.554795,1320.533416,1.449830,-7.370849540736187e-07 [mol/L],1.5124513402886703e-06 [mol/L],8.991643832170565e-06 [mol/L],9.728728786244183e-06 [mol/L]
12,NI,3780.077086,1000.000000,684.082189,125.737464,Minimal,3095.994897,1007.873955,1.449830,3.545947096031513e-06 [mol/L],1.1543519430757217e-06 [mol/L],4.329449437680053e-06 [mol/L],7.835023416485401e-07 [mol/L]
13,ZN,199808.236064,55212.922173,215018.559173,14304.788166,Minimal,-15210.323109,57035.898690,1.449830,-1.7420894688916387e-05 [mol/L],6.532513329550545e-05 [mol/L],0.0002288470937471057 [mol/L],0.0002462679884360221 [mol/L]
